# Day 7: where things stand

A guided look at everything run so far on the **real models** (Llama 3.1 8B on vLLM on a Modal H100; Muse Spark 1.2 through the Meta Model API), and what it tells us before the full runs.

1. **What has been checked**: the bring-up, step by step.
2. **The ingest check**: does concurrent ingest give the same memory as sequential ingest?
3. **Memory quality**: how much summaries differ between runs, and whether concurrency adds to that.
4. **Speed and cost**: what the full runs will take.
5. **Answers item by item**: what's behind the accuracy numbers (and the smoke run's FAIL).
6. **A measurement caveat**: LoCoMo's "evidence in memory" with real summaries.
7. **Spend so far**.
8. **What it suggests, and next steps**.

Set the check to look at in the next cell. It needs local Postgres running (`docker compose start`); everything else is read from `eval/results/`. Nothing here calls a model.

In [ ]:
CHECK_ID = "smoke"   # the CHECK_ID scripts/ingest_check.sh used
N = 5                # and its CHECK_N

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "eval").is_dir() else Path.cwd().parent
os.chdir(ROOT)  # so .env and the repo's packages are found
sys.path.insert(0, str(ROOT))

from eval import ingest_check, locomo, swebench_loc  # noqa: E402
from gateway.config import Settings  # noqa: E402

RESULTS = ROOT / "eval" / "results"
MODES = ["seqa", "seqb", "conc"]
BASE = Settings().database_url.rsplit("/", 1)[0]
pd.set_option("display.max_colwidth", 90)


def summary(name: str) -> dict:
    """A run's summary file (eval/results/<name>.json), or {} if it doesn't exist."""
    path = RESULTS / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else {}


def results(name: str) -> list[dict]:
    """A run's per-item results (eval/results/raw/<name>.jsonl)."""
    path = RESULTS / "raw" / f"{name}.jsonl"
    return [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []


items = swebench_loc.load(N, 0) + locomo.load(N, 0)  # the same items the check used
by_id = {item.id: item for item in items}

## 1. What has been checked

| Step | What it shows | Result |
|---|---|---|
| Simulated full pipeline (Day 6) | The harness, gateway and accounting are right: invariants hold exactly on simulated models | passed |
| `modal deploy` | vLLM serves Llama 3.1 8B on one H100 | working after three fixes (no Hugging Face token on the server, a model-cache check, Transformers 5 vs vLLM 0.11) |
| `scripts.hello` | Both models answer; a 12K-token prompt isn't truncated | passed. Cold start ≈ 80 s. Muse is a reasoning model, so the LoCoMo judge now gets 1,024 tokens at minimal reasoning effort |
| `scripts.cache_check` | A teammate reuses the team's KV cache; another team can't | passed (table below) |
| `scripts/ingest_check.sh`, 5 items (`smoke`) | The sequential vs concurrent comparison runs end to end | ran; one FAIL, looked at in sections 5–6 |
| Harness fixes after the smoke run (30 Oct) | Answers at temperature 0; LoCoMo retrieval measured by session (`source_in_hits`) | done; applies to runs from now on |
| `scripts/ingest_check.sh`, 100 items | The real decision on concurrent ingest | **next** |

**KV cache check** (same 6,150-token team memory, asked by three people):

| Caller | Team | Seconds | Prompt tokens reused from cache |
|---|---|---|---|
| alice | web | 0.59 | 0 (first) |
| bob | web | 0.43 | **6,144** |
| carol | ops | 0.53 | **0** |

Sharing works within a team and never crosses teams. The time saved is small here because an H100 processes 6K tokens in a fraction of a second and most of each request is the round trip to Modal; reused tokens are the measure that matters.

## 2. The ingest check

The same items are ingested into three fresh databases on the same deployment:

- **seqa**: one page at a time (`INGEST_CONCURRENCY=1`), the original plan.
- **seqb**: the same again. It measures **run-to-run noise**: compression samples at temperature 0.2, so even two sequential runs write different summaries.
- **conc**: 16 pages at a time, so vLLM batches them.

Each database then answers the items with `llama_only` (local model only). The rule, fixed before running: **conc passes a measure if it's no further from seqa than seqb is**, plus a small tolerance. The page set must match exactly, since splitting is deterministic.

In [ ]:
rows = []
for bench in ["swebench", "locomo"]:
    for m in MODES:
        ingest = summary(f"{bench}_ingest_{CHECK_ID}-{m}")
        answers = summary(f"{bench}_llama_only_{CHECK_ID}-{m}")
        rows.append({
            "bench": bench, "mode": m,
            "concurrency": ingest.get("ingest_concurrency"),
            "pages": ingest.get("ingest_pages"),
            "fallback rate": ingest.get("ingest_fallback_rate"),
            "ingest minutes": round(ingest.get("ingest_wall_seconds", 0) / 60, 1),
            "pages / s": ingest.get("pages_per_second"),
            "items answered": answers.get("items"),
            "accuracy": answers.get("accuracy"),
            "evidence in memory": answers.get("gold_in_hits_rate"),
        })
overview = pd.DataFrame(rows).set_index(["bench", "mode"])
labels = summary(f"swebench_ingest_{CHECK_ID}-seqa")
display(Markdown(f"Models: **{labels.get('llama_model')}** answering and compressing; "
                 f"simulated: {labels.get('simulated')}."))
overview

How to read it:
- **pages** must be identical across the three modes, or something went wrong (e.g. an interrupted run left a document half stored).
- **fallback rate** is the share of pages where Llama's JSON was unusable twice, so the raw page start was stored instead.
- **pages / s** is ingest throughput, the number that sets the cost of a full run (section 4).
- **accuracy** comes from very few items in a smoke run: with 5 items, one item is 20 points.

## 3. Memory quality: does concurrency change the summaries?

Every page has a summary in each database. For each page, compare its summary in seqb and in conc with its summary in seqa (cosine similarity of the summary embeddings, 1 = same meaning). If concurrency changed the summaries, *conc vs seqa* would sit clearly below *seqb vs seqa*.

In [ ]:
teams = {item.team_id for item in items}
memory = {}
for m in MODES:
    memory[m], _ = await ingest_check.read(f"{BASE}/{ingest_check.database(CHECK_ID, m)}", teams)


def cosines(a: dict, b: dict) -> pd.Series:
    """Per page: cosine between its summary embedding in a and in b."""
    out = {}
    for key, rows in a.items():
        if key in b:
            va, vb = rows[0][2], b[key][0][2]
            out[key] = float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))
    return pd.Series(out)


sim = pd.DataFrame({"seqb vs seqa (noise)": cosines(memory["seqb"], memory["seqa"]),
                    "conc vs seqa": cosines(memory["conc"], memory["seqa"])})
identical = {name: np.mean([memory[m][k][0][0] == memory["seqa"][k][0][0] for k in sim.index])
             for name, m in [("seqb vs seqa (noise)", "seqb"), ("conc vs seqa", "conc")]}
table = sim.describe(percentiles=[0.1, 0.5]).T[["count", "mean", "50%", "10%", "min"]]
table["identical summaries"] = pd.Series(identical)
table.round(3)

One page with a typical difference (the page whose *conc vs seqa* similarity is closest to the median), and its three summaries:

In [ ]:
key = (sim["conc vs seqa"] - sim["conc vs seqa"].median()).abs().idxmin()
team, source, page = key
display(Markdown(f"**Page** from `{source}`, first 500 of {len(page)} characters:"))
print(page[:500])
for m in MODES:
    display(Markdown(f"**{m}:** {memory[m][key][0][0]}"))

## 4. Speed, and what the full runs will take

Throughput is total pages over total ingest wall time for each mode. The page counts of a full run come from the simulated full run, which ingests the same documents.

A 5-item run is a rough guide: with few documents, the concurrent mode can't always keep 16 calls in flight, so the 100-item check gives the better estimate.

In [ ]:
H100_PER_HOUR = 3.95  # Modal's listed H100 price
FULL_RUN_PAGES = {"benchmarks (300 SWE-bench + 300 LoCoMo)": 6615, "team replay (4 runs)": 6633}

speed = {}
for m in MODES:
    pages = sum(summary(f"{b}_ingest_{CHECK_ID}-{m}").get("ingest_pages", 0) for b in ["swebench", "locomo"])
    wall = sum(summary(f"{b}_ingest_{CHECK_ID}-{m}").get("ingest_wall_seconds", 0) for b in ["swebench", "locomo"])
    speed[m] = pages / wall if wall else float("nan")
display(Markdown(f"Pages per second: seqa **{speed['seqa']:.2f}**, seqb **{speed['seqb']:.2f}**, "
                 f"conc **{speed['conc']:.2f}**, so concurrent ingest is "
                 f"**{speed['conc'] / speed['seqa']:.1f}x** faster."))

rows = []
for what, pages in FULL_RUN_PAGES.items():
    for mode, rate in [("sequential", speed["seqa"]), ("concurrent (16)", speed["conc"])]:
        hours = pages / rate / 3600
        rows.append({"ingest": what, "mode": mode, "pages": pages,
                     "hours": round(hours, 1), "H100 $": round(hours * H100_PER_HOUR, 1)})
pd.DataFrame(rows).set_index(["ingest", "mode"])

This is ingest only. Answering adds a few hundred to a few thousand Llama calls per run (short, and cached when memory is shared), and every session adds a cold start (~80 s) plus up to 5 idle minutes before Modal scales to zero.

## 5. Answers item by item

Each row is one item; each mode column shows ✓/✗ and the start of Llama's answer. In the smoke run Llama's answers were **sampled** (the harness didn't set a temperature, so vLLM used the model's default of 0.6), so the same question with the same memory could get a different answer on another run. That, plus small samples, is what moved accuracy. Runs from 30 Oct on ask for temperature 0, so accuracy differences come from memory; each result file records it as `answer_temperature`.

In [ ]:
def item_table(bench: str) -> pd.DataFrame:
    rows = {}
    for m in MODES:
        for r in results(f"{bench}_llama_only_{CHECK_ID}-{m}"):
            item = by_id[r["item_id"]]
            row = rows.setdefault(r["item_id"], {
                "question": item.meta.get("question", "(the issue text)")[:60],
                "gold": ", ".join(item.gold) if isinstance(item.gold, list) else str(item.gold)})
            ok = bool(r["score"].get("hit", r["score"].get("correct")))
            row[m] = ("✓ " if ok else "✗ ") + r["answer"].replace("\n", " ")[:70]
    return pd.DataFrame.from_dict(rows, orient="index")


temperatures = {m: summary(f"locomo_llama_only_{CHECK_ID}-{m}").get("answer_temperature", "0.6 (model default)")
                for m in MODES}
display(Markdown(f"Answer temperature by mode: {temperatures}"))
display(Markdown("**LoCoMo** (questions about long conversations, graded by Muse)"))
display(item_table("locomo"))
display(Markdown("**SWE-bench** (which files a fix touches; scored against the real patch)"))
display(item_table("swebench"))

## 6. A measurement caveat: LoCoMo's "evidence in memory"

`gold_in_hits_rate` asks whether the **exact** conversation turn holding the answer appears in the memory summaries the gateway retrieved. Real Llama summaries paraphrase, so this is close to 0 by construction (on the simulator, summaries start with the page's own text, which is why it was non-zero there). It says little about retrieval on real models.

A looser check that does work: did the gateway retrieve a memory **from the conversation session** that holds the evidence turn? (Sources are session labels, and a session can span several pages, so this is an upper bound on "the evidence was shown".) Runs from 30 Oct record this per item as `source_in_hits`, and `eval.ingest_check` compares it across modes as `locomo_source_in_hits_rate`; the cell below works it out from the retrieved sources, so it covers older runs too.

In [ ]:
def session_retrieved(r: dict) -> bool | None:
    """Was memory from every session holding the evidence retrieved? (source_in_hits)"""
    item = by_id[r["item_id"]]
    if not item.evidence_sources:
        return None
    return set(item.evidence_sources) <= set(r["retrieved_sources"])


rows = []
for m in MODES:
    rs = results(f"locomo_llama_only_{CHECK_ID}-{m}")
    found = [session_retrieved(r) for r in rs]
    found = [f for f in found if f is not None]
    rows.append({"mode": m, "items": len(rs),
                 "exact turn in retrieved summaries": np.mean([bool(r["gold_in_hits"]) for r in rs]),
                 "evidence session retrieved": np.mean(found) if found else None,
                 "accuracy": np.mean([bool(r["score"].get("correct")) for r in rs])})
pd.DataFrame(rows).set_index("mode").round(3)

## 7. Spend so far

From each check database's usage ledger: every model call with its exact token counts. Muse calls here are LoCoMo judge calls; Llama calls are free per call (we pay for the GPU's time instead, section 4).

Modal: the dashboard showed **$8.63** after the first deploy, almost all of it a crash loop (vLLM failed to start and each container sat on the H100 for up to 10 minutes). The deployment now ends a container the moment vLLM exits.

In [ ]:
rows = []
for m in MODES:
    with psycopg.connect(f"{BASE}/{ingest_check.database(CHECK_ID, m)}") as conn:
        for model, route, calls, prompt, completion, usd in conn.execute(
                "SELECT model, route, count(*), sum(prompt_tokens), sum(completion_tokens),"
                " sum(cost_usd) FROM usage GROUP BY 1, 2 ORDER BY 1, 2"):
            rows.append({"mode": m, "model": model, "route": route, "calls": calls,
                         "prompt tokens": prompt, "completion tokens": completion,
                         "USD": float(usd)})
spend = pd.DataFrame(rows)
display(spend.set_index(["mode", "model", "route"]))
display(Markdown(f"Muse total across the three databases: **${spend.USD.sum():.4f}**"))

## 8. What it suggests, and next steps

**From the 5-item smoke run (29 Oct):**

- **The whole path works on real models**: the three databases hold exactly the same 347 pages, compression almost never falls back, and the Muse judge returns verdicts.
- **Concurrency doesn't visibly change memory.** Summaries differ between *any* two runs (only ~10% are word-for-word identical even between the two sequential runs), and conc differs from seqa by the same amount as seqb does (section 3).
- **Concurrent ingest is much faster**: about 5–9x on these items, which turns a multi-hour sequential ingest into well under an hour (section 4).
- **The FAIL on LoCoMo accuracy is not evidence against concurrency.** It rests on 5 questions where one answer is 20 points; the answers are sampled; and the evidence measure can't see paraphrased memory (sections 5–6). The memory measures, which are the ones concurrency could affect, all pass, and conc's memory actually led to the evidence's session more often (2 of 5 items vs 1 of 5).

**Done since (30 Oct):** answers are requested at temperature 0, and LoCoMo retrieval is measured by session (`source_in_hits`, sections 5–6). Both apply to the 100-item check and the full runs, not to the smoke results above.

**Next:**

1. Run the 100-item check: `caffeinate -i scripts/ingest_check.sh`, then set `CHECK_ID = "check1"` and `N = 100` here and rerun this notebook. The bullets above describe the smoke run; check them against the new numbers.
2. If its memory measures pass, run the full eval with concurrent ingest: `caffeinate -i scripts/full_run.sh` (`INGEST_CONCURRENCY=16` by default). If they fail, run it at `INGEST_CONCURRENCY=1`, the original plan.
3. Optional, if time and credit allow: the sequential reference, `FULL_ID=seq1 INGEST_CONCURRENCY=1 caffeinate -i scripts/full_run.sh`. It repeats the Muse spend.
